# Phan's three-layer lattice on a periodic quotient

Upload this notebook to Colab, choose **Runtime → Change runtime type → GPU**, and run the code cells in order. They use the runtime's PyTorch and fall back to CPU when CUDA is unavailable.

The quotient reduces the first $n-1$ coordinates modulo a common period $M\ge3$ and keeps the sum of coordinates as a separate layer label. At edge index $i$ an outer vertex meets the middle layer by adding $e_i$ from below or subtracting $e_i$ from above. Every four-cycle specifies a lower vertex $a$ and a pair $i<j$. Its upper vertex is $a+e_i+e_j$. The per-coordinate net displacement of a closed four-step walk with two upward and two downward steps lies between $-2$ and $2$, so reduction modulo $M\ge3$ cannot add a spurious four-cycle.

The search runs many independent annealing chains on the device. A move adds one missing edge and removes one other edge from each cycle it would complete. Each reported density comes with a verified periodic pattern. The counting ceiling is $3/4$ before rounding, so dimension 5 is the first where the conjectured threshold drops below that ceiling.

In [ ]:
import torch
from itertools import combinations, product
from math import sqrt
from time import perf_counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU")


In [ ]:
def periodic_problem(n, period):
    """Return edge count and the list of all distinct cycle constraints."""
    if n < 2 or period < 3:
        raise ValueError("Require n >= 2 and period >= 3")
    sites = tuple(product(range(period), repeat=n - 1))
    site_id = {site: i for i, site in enumerate(sites)}
    pairs = tuple(combinations(range(n), 2))
    half = n * len(sites)
    cycles = []
    for a_id, a in enumerate(sites):
        for i, j in pairs:
            b = tuple((v + (k == i) + (k == j)) % period for k, v in enumerate(a))
            minus = n * a_id
            plus = half + n * site_id[b]
            cycles.append((minus + i, minus + j, plus + i, plus + j))
    return 2 * half, cycles


def verify_graph(n, period, kept):
    """Independent check by enumerating neighbor pairs in the quotient graph."""
    sites = tuple(product(range(period), repeat=n - 1))
    half = n * len(sites)
    common = set()
    for side in range(2):
        sign = 1 if side == 0 else -1
        for a_id, a in enumerate(sites):
            base = side * half + n * a_id
            neighbors = [
                tuple((v + sign * (k == i)) % period for k, v in enumerate(a))
                for i in range(n) if kept[base + i]
            ]
            for pair in combinations(sorted(neighbors), 2):
                if pair in common:
                    raise RuntimeError(f"Unexpected 4-cycle through middle sites {pair}")
                common.add(pair)

In [ ]:
def build_problem(n, period, device):
    edge_count, cycles = periodic_problem(n, period)
    adjacent = [[] for _ in range(edge_count)]
    for cycle in cycles:
        for edge in cycle:
            adjacent[edge].append(tuple(e for e in cycle if e != edge))
    for neighbors in adjacent:
        flat = [edge for triple in neighbors for edge in triple]
        if len(neighbors) != n - 1 or len(flat) != len(set(flat)):
            raise RuntimeError("Unexpected overlap in cycles through one edge")
    return (edge_count, torch.tensor(cycles, dtype=torch.long, device=device),
            torch.tensor(adjacent, dtype=torch.long, device=device))


def contains_cycle(states, cycles, chunk=1024):
    invalid = torch.zeros(states.shape[0], dtype=torch.bool, device=states.device)
    for block in cycles.split(chunk):
        invalid |= states[:, block].all(dim=-1).any(dim=-1)
    return invalid


@torch.no_grad()
def anneal(n, period, device, *, batch, steps, seed=507835, checkpoint=8):
    if min(batch, steps, checkpoint) <= 0:
        raise ValueError("batch, steps and checkpoint must be positive")
    edge_count, cycles, adjacent = build_problem(n, period, device)
    rng = torch.Generator(device=device).manual_seed(seed)
    rows = torch.arange(batch, device=device)
    edge_ids = torch.arange(edge_count, device=device)
    # Alternate starting with the full lower or upper layer, with independent RNG.
    kept = ((edge_ids >= edge_count // 2)[None, :] == (rows % 2 == 0)[:, None])
    counts = kept.sum(dim=1)
    best_counts = counts.clone()
    best = kept.clone()

    for step in range(steps):
        added = torch.randint(edge_count, (batch,), device=device, generator=rng)
        neighbors = adjacent[added]
        blockers = kept[rows[:, None, None], neighbors].all(dim=-1)
        choices = torch.randint(3, (batch, n - 1, 1), device=device, generator=rng)
        evict = neighbors.gather(2, choices).squeeze(2)
        delta = 1 - blockers.sum(dim=1)
        temperature = 0.10 + 0.75 * (1 - step / steps)
        draw = torch.rand(batch, device=device, generator=rng)
        accept = (~kept[rows, added]) & (
            (delta >= 0) | (draw < torch.exp(delta / temperature))
        )

        kept[rows, added] = kept[rows, added] | accept
        kept[rows[:, None], evict] = (
            kept[rows[:, None], evict] & ~(blockers & accept[:, None])
        )
        counts += torch.where(accept, delta, 0)
        if (step + 1) % checkpoint == 0 or step + 1 == steps:
            improve = counts > best_counts
            best = torch.where(improve[:, None], kept, best)
            best_counts = torch.maximum(best_counts, counts)

    if (not torch.equal(counts, kept.sum(dim=1))
            or not torch.equal(best_counts, best.sum(dim=1))
            or bool(contains_cycle(kept, cycles).any())
            or bool(contains_cycle(best, cycles).any())):
        raise RuntimeError("Tensor cycle verification failed")
    index = int(best_counts.argmax().item())
    witness = best[index].cpu().tolist()
    verify_graph(n, period, witness)
    upper = edge_count - (len(cycles) + n - 2) // (n - 1)
    return witness, upper


In [ ]:
# A deliberately invalid mask and a one-layer feasible mask test the checker.
edges, cycles, _ = build_problem(3, 3, device)
examples = torch.ones((2, edges), dtype=torch.bool, device=device)
examples[1, edges // 2:] = False
assert contains_cycle(examples, cycles).tolist() == [True, False]

configs = [(4, 4), (5, 4)]
batch = 192 if device.type == "cuda" else 8
steps = 40000 if device.type == "cuda" else 20000

witnesses = {}
for n, period in configs:
    started = perf_counter()
    witness, upper = anneal(n, period, device, batch=batch, steps=steps)
    witnesses[n, period] = witness
    density = sum(witness) / len(witness)
    conjectured = (1 + 1 / sqrt(n)) / 2
    print(f"n={n}, period={period}, kept={sum(witness)}/{len(witness)}, "
          f"density={density:.6f}, counting upper={upper/len(witness):.6f}, "
          f"conjectured={conjectured:.6f}, elapsed={perf_counter()-started:.2f}s")


Each `witnesses[n, period]` is a Boolean mask in lexicographic residue order, with the lower layer first and direction indices innermost. For larger GPU runs, raise the period gradually. A batch stores on the order of `batch * 2 * n * period**(n-1)` state bits as one-byte Boolean elements plus scratch buffers and cycle tensors. Increasing the batch explores more parallel trajectories. Increasing the number of steps gives each trajectory more time to improve. The recorded best is sampled at checkpoints to limit whole-state copies.